# wandb-watch-model — ex1: watch the trainable head with parameter+gradient logging

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wandb-watch-model`. Running the final beacon cell reports progress against the `Logging: wandb.watch model` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.watch model` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-watch-model`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-watch-model"
DD_SUBTOPIC = "Logging: wandb.watch model"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `wandb.watch(module, log='all', log_freq=K)` — quick refresher

`wandb.watch(module, log='all', log_freq=K)` hooks the named module so wandb auto-logs its **parameters** AND **gradients** every K forward passes. You point it at a single submodule (or `self.model`) and the wandb dashboard grows histogram plots for each tracked tensor.

**The three kwargs:**

- First positional: the `nn.Module` (or list of modules) to watch.
- `log='all'` — log both parameters AND gradients. Other options are `'parameters'`, `'gradients'`, or `None` (disable).
- `log_freq=K` — log every K forward passes. ARENA's guidance: make this LESS than 1 epoch's worth of steps, otherwise the dashboard shows zero histograms.

**Call once per run**, inside `pre_training_setup` after `wandb.init` and after the model is on the right device. Calling it twice on the same module duplicates the hooks.

**ARENA's choice of submodule.** Watching `self.model` itself logs every layer's weights — useful but noisy. ARENA's resnet-fine-tune example watches just the head (`self.model.out_layers[-1]`) because that's the only module being trained.

### Exercise 1 — watch the trainable head with parameter+gradient logging

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `wandb.watch(module, log='all', log_freq=K)` to a specific submodule (the trainable head) inside a fake setup function, verified by mocking the `wandb` module.
> Keywords: wandb, watch, histogram, mock
> ```

**KCs targeted:** `wandb-watch-target-module`, `wandb-watch-log-all-freq`

Implement `ex1_attach_wandb_watch(model, log_freq)`. The canonical ARENA `pre_training_setup` line for histogram logging:

1. `model` is an `nn.Module` with a `.out_layers` `nn.ModuleList`. The HEAD is `model.out_layers[-1]` — that's the module we want to watch (not the whole model — too noisy).
2. Call `wandb.watch(...)` with EXACTLY these args:
   - First positional: `model.out_layers[-1]` (the head module).
   - `log='all'` (log both parameters AND gradients).
   - `log_freq=log_freq` (passed through as a kwarg).
3. Return the head module that was watched (for the caller to sanity-check).

The test mocks wandb and inspects `wandb.watch.call_args` to verify the exact module reference and the kwargs.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb
import torch as t

def ex1_attach_wandb_watch(model, log_freq):
    head = model.out_layers[-1]
    wandb.watch(head, log='all', log_freq=log_freq)
    return head


<details><summary>Solution</summary>

```python
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb
import torch as t

def ex1_attach_wandb_watch(model, log_freq):
    head = model.out_layers[-1]
    wandb.watch(head, log='all', log_freq=log_freq)
    return head
```

**Why watch the head, not the whole model.** ARENA's resnet fine-tune freezes everything except the final classifier. `wandb.watch(self.model, ...)` would attach hooks to every frozen layer — they'd record zero-gradient histograms and waste dashboard real estate. Watching just `out_layers[-1]` keeps the histograms relevant.

**`log='all'` is the kitchen sink.** Wandb supports `'parameters'` (weights only), `'gradients'` (gradients only), or `'all'` (both). ARENA picks `'all'` — for fine-tuning you want to see both the weights drift AND the gradient magnitudes evolve.

**`log_freq` units = forward passes, not training steps.** They happen to coincide in supervised training (one forward per step), but if you have an inner-loop training algorithm with multiple forwards per step, plan accordingly.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()